# 05 — Selección de Modelo

## ¿Por qué detección de anomalías no supervisada?

Nuestro problema tiene una restricción fundamental: **no tenemos etiquetas**. No existe un dataset con meses marcados como "anomalía" o "normal" por un experto. Solo tenemos datos crudos y conocimiento general de cuándo ocurrieron crisis.

Esto descarta modelos supervisados (clasificación, regresión) y nos deja en el territorio de **detección de anomalías no supervisada**.

## Candidatos a evaluar

| Modelo | Tipo | Ventajas | Desventajas |
|--------|------|----------|-------------|
| **Isolation Forest** | Basado en árboles | Rápido, escalable, maneja bien alta dimensionalidad | Sensible a contamination |
| **Local Outlier Factor** | Basado en densidad | Detecta anomalías locales | Lento en datasets grandes, no predice nuevos datos |
| **One-Class SVM** | Basado en frontera | Bueno con pocas features | Muy lento con muchas features, requiere tunear kernel |
| **DBSCAN** | Clustering | No requiere # clusters | Sensible a epsilon, no diseñado para anomaly scoring |

Vamos a comparar los 3 principales en nuestros datos.

In [ ]:
import sys, time, warnings
sys.path.insert(0, '../..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.processing.features import engineer_features

# Cargar y preparar datos
df = pd.read_csv('../../data/raw/ecuador_electricity_real.csv', parse_dates=['fecha'])
df.columns = [c.replace(',', '_').replace(' ', '_') for c in df.columns]

value_cols = ['gen_bioenergy', 'gen_gas', 'gen_hydro', 'gen_other_fossil', 'gen_solar',
              'gen_total_generation', 'gen_wind', 'demanda_twh', 'co2_intensity_gco2_kwh',
              'importaciones_netas_twh']
value_cols = [c for c in value_cols if c in df.columns]

df_feat = engineer_features(df.copy(), date_col='fecha', value_cols=value_cols)

# Preparar features numéricos
exclude = ['fecha', 'anio', '_source_file']
feature_cols = [c for c in df_feat.select_dtypes(include=[np.number]).columns 
                if not any(x in c for x in exclude)]

X = df_feat[feature_cols].fillna(df_feat[feature_cols].median())
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"Features para comparación: {X_scaled.shape}")

## 5.1 Comparación de modelos

Entrenamos cada modelo con contamination=8% y comparamos:
1. **Qué meses marca como anomalía** — ¿coincide con crisis reales?
2. **Tiempo de entrenamiento**
3. **Capacidad de scoring** — ¿produce un score continuo o solo binario?

In [ ]:
# Crisis oct-dic 2024 (meses que DEBERÍAN ser detectados)
crisis_mask = (df_feat['fecha'] >= '2024-10-01') & (df_feat['fecha'] <= '2024-12-31')
crisis_indices = set(df_feat[crisis_mask].index)

results = {}

# 1. Isolation Forest
t0 = time.time()
iso = IsolationForest(n_estimators=300, contamination=0.08, random_state=42, n_jobs=-1)
iso_pred = iso.fit_predict(X_scaled)
iso_scores = iso.decision_function(X_scaled)
iso_time = time.time() - t0
iso_anomalies = set(np.where(iso_pred == -1)[0])
results['Isolation Forest'] = {
    'predictions': iso_pred, 'scores': iso_scores, 'time': iso_time,
    'anomalies': iso_anomalies, 'has_scoring': True
}

# 2. Local Outlier Factor
t0 = time.time()
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.08)
lof_pred = lof.fit_predict(X_scaled)
lof_scores = lof.negative_outlier_factor_
lof_time = time.time() - t0
lof_anomalies = set(np.where(lof_pred == -1)[0])
results['LOF'] = {
    'predictions': lof_pred, 'scores': lof_scores, 'time': lof_time,
    'anomalies': lof_anomalies, 'has_scoring': True
}

# 3. One-Class SVM
t0 = time.time()
svm = OneClassSVM(nu=0.08, kernel='rbf', gamma='scale')
svm_pred = svm.fit_predict(X_scaled)
svm_scores = svm.decision_function(X_scaled)
svm_time = time.time() - t0
svm_anomalies = set(np.where(svm_pred == -1)[0])
results['One-Class SVM'] = {
    'predictions': svm_pred, 'scores': svm_scores, 'time': svm_time,
    'anomalies': svm_anomalies, 'has_scoring': True
}

# Tabla comparativa
comparison = []
for name, r in results.items():
    crisis_detected = len(r['anomalies'] & crisis_indices)
    comparison.append({
        'Modelo': name,
        'Anomalías totales': len(r['anomalies']),
        'Crisis oct-dic detectadas': f"{crisis_detected}/3",
        'Match crisis (%)': f"{crisis_detected/3*100:.0f}%",
        'Tiempo (s)': f"{r['time']:.3f}",
        'Scoring continuo': '✓' if r['has_scoring'] else '✗',
        'Predice nuevos datos': '✓' if name != 'LOF' else '✗',
    })

pd.DataFrame(comparison)

In [ ]:
# Visualizar qué meses detecta cada modelo
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=['Isolation Forest', 'Local Outlier Factor', 'One-Class SVM'])

for i, (name, r) in enumerate(results.items(), 1):
    colors = ['red' if p == -1 else '#2196F3' for p in r['predictions']]
    fig.add_trace(go.Scatter(
        x=df_feat['fecha'], y=r['scores'], mode='markers+lines',
        marker=dict(color=colors, size=6), line=dict(color='lightgray', width=1),
        name=name), row=i, col=1)
    # Marcar zona de crisis
    fig.add_vrect(x0='2024-10-01', x1='2024-12-31', fillcolor='rgba(255,0,0,0.1)', 
                  row=i, col=1)

fig.update_layout(template='plotly_white', height=600, showlegend=False,
                  title_text='Comparación de Scores por Modelo (rojo = anomalía)')
fig.show()

## 5.2 Decisión: Isolation Forest

**Elegimos Isolation Forest** por las siguientes razones:

| Criterio | IF | LOF | OC-SVM |
|----------|----|----|--------|
| Detecta crisis 2024 | ✓ | Parcial | Variable |
| Puede predecir nuevos datos | ✓ | ✗ | ✓ |
| Scoring continuo | ✓ | ✓ | ✓ |
| Velocidad | Rápido | Medio | Lento |
| Maneja 213 features | Bien | Bien | Mal (curse of dimensionality) |
| Compatible con SHAP | ✓ (TreeExplainer) | ✗ | ✗ |
| Interpretable | Alto | Bajo | Bajo |

### ¿Cómo funciona Isolation Forest?

La intuición es brillante por simple: **las anomalías son fáciles de aislar**.

1. Selecciona una feature aleatoria
2. Selecciona un valor de corte aleatorio
3. Repite recursivamente → construye un árbol
4. Los puntos anómalos necesitan **menos cortes** para quedar solos
5. El score es inversamente proporcional a la profundidad promedio

Un mes con generación hidro inusualmente baja Y fósil inusualmente alta Y demanda cayendo se aísla rápidamente → score bajo → anomalía.

---
*Siguiente notebook: 06 — Entrenamiento y Evaluación Final*